# 🚬 흡연 분류 AI 해커톤 - V2 (정수 출력 버전)

**목표:** ROC-AUC 0.77+ 달성

**⚠️ 중요:** 제출 파일의 label은 **0 또는 1 정수**로 출력

---

## 📌 STEP 1: 환경 설정

In [ ]:
# 1-1. 라이브러리 설치
!pip install -q xgboost lightgbm catboost

In [ ]:
# 1-2. Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 1-3. 경로 설정 (본인 경로에 맞게 수정하세요)
base_path = '/content/drive/MyDrive/AI_Projects/smoking_hackathon/'
train_path = base_path + 'data/train.csv'
test_path = base_path + 'data/test.csv'
submission_path = base_path + 'data/sample_submission.csv'
result_path = base_path + 'results/'

In [ ]:
# 1-4. 라이브러리 임포트
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# 머신러닝
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.model_selection import RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression

# 부스팅 모델
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# 시드 고정
import random
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
set_seed(42)

print("✅ 라이브러리 임포트 완료!")

In [ ]:
# 1-5. 한글 폰트 설정 (Colab)
!apt-get update -qq
!apt-get install -qq fonts-nanum*

fe = fm.FontEntry(
    fname="/usr/share/fonts/truetype/nanum/NanumGothic.ttf",
    name="NanumGothic"
)
fm.fontManager.ttflist.insert(0, fe)
plt.rcParams.update({"font.size": 10, "font.family": "NanumGothic"})
plt.rcParams["axes.unicode_minus"] = False

print("✅ 한글 폰트 설정 완료!")

## 📌 STEP 2: 데이터 로드

In [ ]:
# 2-1. 데이터 로드
train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
submission = pd.read_csv(submission_path)

print("=" * 50)
print("📊 데이터 기본 정보")
print("=" * 50)
print(f"Train: {train.shape}")
print(f"Test: {test.shape}")
print(f"\n컬럼: {train.columns.tolist()}")

print(f"\n🎯 타겟 분포:")
print(train['label'].value_counts())
print(f"흡연자 비율: {train['label'].mean()*100:.2f}%")

## 📌 STEP 3: 데이터 전처리

In [ ]:
# 3-1. 원본 복사
train_df = train.copy()
test_df = test.copy()

# 3-2. ID 분리 및 제거
if 'ID' in test_df.columns:
    test_id = test_df['ID'].copy()
elif 'id' in test_df.columns:
    test_id = test_df['id'].copy()
else:
    test_id = None

train_df = train_df.drop(['ID', 'id'], axis=1, errors='ignore')
test_df = test_df.drop(['ID', 'id'], axis=1, errors='ignore')

# 3-3. 특성과 타겟 분리
X = train_df.drop('label', axis=1, errors='ignore')
y = train_df['label']
X_test = test_df.drop('label', axis=1, errors='ignore')

# 3-4. 컬럼명 저장
feature_cols = X.columns.tolist()
print(f"특성 수: {len(feature_cols)}")
print(f"특성: {feature_cols}")

## 📌 STEP 4: 피처 엔지니어링 (강화 버전)

In [ ]:
def create_features_v2(df):
    """
    개선된 피처 엔지니어링 V2
    """
    df = df.copy()
    
    # 컬럼명 소문자로 매핑
    col_lower = {col: col.lower() for col in df.columns}
    df_l = df.rename(columns=col_lower)
    cols = df_l.columns.tolist()
    
    # ===== 1. 콜레스테롤 관련 비율 =====
    if 'hdl' in cols and 'ldl' in cols:
        df['HDL_LDL_ratio'] = df_l['hdl'] / (df_l['ldl'] + 1)
        df['LDL_HDL_ratio'] = df_l['ldl'] / (df_l['hdl'] + 1)
    
    if 'hdl' in cols and 'cholesterol' in cols:
        df['HDL_Chol_ratio'] = df_l['hdl'] / (df_l['cholesterol'] + 1)
        df['Atherogenic_idx'] = (df_l['cholesterol'] - df_l['hdl']) / (df_l['hdl'] + 1)
    
    if 'triglyceride' in cols and 'hdl' in cols:
        df['TG_HDL_ratio'] = df_l['triglyceride'] / (df_l['hdl'] + 1)
    
    # ===== 2. 혈압 관련 =====
    if 'systolic' in cols and 'diastolic' in cols:
        df['Pulse_pressure'] = df_l['systolic'] - df_l['diastolic']
        df['MAP'] = df_l['diastolic'] + (df_l['systolic'] - df_l['diastolic']) / 3
        df['BP_ratio'] = df_l['systolic'] / (df_l['diastolic'] + 1)
    
    # ===== 3. 간 기능 (흡연과 강한 연관) =====
    if 'gtp' in cols:
        df['GTP_log'] = np.log1p(df_l['gtp'])
        df['GTP_sqrt'] = np.sqrt(df_l['gtp'])
        df['GTP_sq'] = df_l['gtp'] ** 2
    
    if 'ast' in cols and 'alt' in cols:
        df['AST_ALT_ratio'] = df_l['ast'] / (df_l['alt'] + 1)
        df['Liver_sum'] = df_l['ast'] + df_l['alt']
    
    # ===== 4. 헤모글로빈 (흡연자가 높음) =====
    if 'hemoglobin' in cols:
        df['Hemo_sq'] = df_l['hemoglobin'] ** 2
        df['Hemo_log'] = np.log1p(df_l['hemoglobin'])
    
    # ===== 5. 시력 관련 =====
    eye_cols = [c for c in cols if 'eyesight' in c]
    if len(eye_cols) >= 2:
        df['Eyesight_avg'] = df_l[eye_cols].mean(axis=1)
        df['Eyesight_diff'] = abs(df_l[eye_cols[0]] - df_l[eye_cols[1]])
    
    # ===== 6. 청력 관련 =====
    hear_cols = [c for c in cols if 'hearing' in c]
    if len(hear_cols) >= 2:
        df['Hearing_sum'] = df_l[hear_cols].sum(axis=1)
    
    # ===== 7. 나이 상호작용 =====
    if 'age' in cols:
        age = df_l['age']
        if 'hemoglobin' in cols:
            df['Age_x_Hemo'] = age * df_l['hemoglobin']
        if 'gtp' in cols:
            df['Age_x_GTP'] = age * df_l['gtp']
        if 'triglyceride' in cols:
            df['Age_x_TG'] = age * df_l['triglyceride']
        if 'hdl' in cols:
            df['Age_x_HDL'] = age * df_l['hdl']
        
        df['Age_sq'] = age ** 2
        df['Age_group'] = pd.cut(age, bins=[0,30,40,50,60,100], labels=[0,1,2,3,4]).astype(float)
    
    # ===== 8. 중성지방 관련 =====
    if 'triglyceride' in cols:
        df['TG_log'] = np.log1p(df_l['triglyceride'])
        df['TG_sq'] = df_l['triglyceride'] ** 2
    
    # ===== 9. 혈당 관련 =====
    fbs_cols = [c for c in cols if 'blood' in c or 'fasting' in c or 'sugar' in c]
    if len(fbs_cols) > 0:
        fbs = df_l[fbs_cols[0]]
        df['FBS_log'] = np.log1p(fbs)
    
    # ===== 10. 건강 지표 통계 =====
    health_cols = [c for c in cols if c in ['systolic','diastolic','hemoglobin','triglyceride','cholesterol','hdl','ldl']]
    if len(health_cols) >= 3:
        df['Health_mean'] = df_l[health_cols].mean(axis=1)
        df['Health_std'] = df_l[health_cols].std(axis=1)
        df['Health_max'] = df_l[health_cols].max(axis=1)
        df['Health_min'] = df_l[health_cols].min(axis=1)
    
    # 결측치, 무한값 처리
    df = df.fillna(0)
    df = df.replace([np.inf, -np.inf], 0)
    
    return df

# 피처 엔지니어링 적용
print("피처 엔지니어링 적용 중...")
X_fe = create_features_v2(X)
X_test_fe = create_features_v2(X_test)

print(f"\n✅ 피처 엔지니어링 완료!")
print(f"원본: {len(feature_cols)}개 → 새로운: {X_fe.shape[1]}개")

## 📌 STEP 5: 이상치 처리 및 스케일링

In [ ]:
# 5-1. 이상치 클리핑
def clip_outliers(df, multiplier=3.0):
    df = df.copy()
    for col in df.columns:
        if df[col].dtype in ['float64', 'int64']:
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            df[col] = df[col].clip(lower=Q1-multiplier*IQR, upper=Q3+multiplier*IQR)
    return df

X_clipped = clip_outliers(X_fe)
X_test_clipped = clip_outliers(X_test_fe)

# 5-2. Train/Val 분할
X_train, X_val, y_train, y_val = train_test_split(
    X_clipped, y, test_size=0.2, random_state=42, stratify=y
)

# 5-3. 스케일링
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test_clipped)

# 전체 데이터용
scaler_full = StandardScaler()
X_full_scaled = scaler_full.fit_transform(X_clipped)
X_test_final = scaler_full.transform(X_test_clipped)

print(f"Train: {X_train_scaled.shape}, Val: {X_val_scaled.shape}, Test: {X_test_final.shape}")
print("✅ 전처리 완료!")

## 📌 STEP 6: 하이퍼파라미터 튜닝

In [ ]:
# 6-1. XGBoost 튜닝
print("🔧 XGBoost 튜닝 중...")

xgb_params = {
    'n_estimators': [300, 500, 700],
    'max_depth': [3, 4, 5, 6],
    'learning_rate': [0.01, 0.02, 0.03, 0.05],
    'min_child_weight': [1, 3, 5, 7],
    'subsample': [0.6, 0.7, 0.8],
    'colsample_bytree': [0.6, 0.7, 0.8],
    'gamma': [0, 0.1, 0.2],
    'reg_alpha': [0, 0.1, 0.5],
    'reg_lambda': [1, 2, 5]
}

xgb_search = RandomizedSearchCV(
    XGBClassifier(random_state=42, verbosity=0, use_label_encoder=False, eval_metric='auc'),
    xgb_params, n_iter=80, cv=5, scoring='roc_auc', random_state=42, n_jobs=-1, verbose=1
)
xgb_search.fit(X_full_scaled, y)
print(f"XGBoost 최고 점수: {xgb_search.best_score_:.5f}")

In [ ]:
# 6-2. LightGBM 튜닝
print("🔧 LightGBM 튜닝 중...")

lgb_params = {
    'n_estimators': [300, 500, 700],
    'max_depth': [3, 5, 7, -1],
    'learning_rate': [0.01, 0.02, 0.03, 0.05],
    'num_leaves': [15, 31, 63],
    'min_child_samples': [10, 20, 30, 50],
    'subsample': [0.6, 0.7, 0.8],
    'colsample_bytree': [0.6, 0.7, 0.8],
    'reg_alpha': [0, 0.1, 0.5],
    'reg_lambda': [0, 0.1, 0.5]
}

lgb_search = RandomizedSearchCV(
    LGBMClassifier(random_state=42, verbose=-1),
    lgb_params, n_iter=80, cv=5, scoring='roc_auc', random_state=42, n_jobs=-1, verbose=1
)
lgb_search.fit(X_full_scaled, y)
print(f"LightGBM 최고 점수: {lgb_search.best_score_:.5f}")

In [ ]:
# 6-3. CatBoost 튜닝
print("🔧 CatBoost 튜닝 중...")

cat_params = {
    'n_estimators': [300, 500, 700],
    'max_depth': [4, 5, 6, 7],
    'learning_rate': [0.01, 0.02, 0.03, 0.05],
    'l2_leaf_reg': [1, 3, 5, 7]
}

cat_search = RandomizedSearchCV(
    CatBoostClassifier(random_state=42, verbose=0),
    cat_params, n_iter=50, cv=5, scoring='roc_auc', random_state=42, n_jobs=-1, verbose=1
)
cat_search.fit(X_full_scaled, y)
print(f"CatBoost 최고 점수: {cat_search.best_score_:.5f}")

In [ ]:
# 6-4. Random Forest 튜닝
print("🔧 Random Forest 튜닝 중...")

rf_params = {
    'n_estimators': [300, 500, 700],
    'max_depth': [10, 15, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

rf_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    rf_params, n_iter=50, cv=5, scoring='roc_auc', random_state=42, n_jobs=-1, verbose=1
)
rf_search.fit(X_full_scaled, y)
print(f"Random Forest 최고 점수: {rf_search.best_score_:.5f}")

In [ ]:
# 6-5. 튜닝 결과 요약
print("\n" + "=" * 50)
print("📊 튜닝 결과 요약")
print("=" * 50)
print(f"XGBoost:  {xgb_search.best_score_:.5f}")
print(f"LightGBM: {lgb_search.best_score_:.5f}")
print(f"CatBoost: {cat_search.best_score_:.5f}")
print(f"RF:       {rf_search.best_score_:.5f}")

## 📌 STEP 7: 최적 임계값 탐색 ⭐

**중요:** 0 또는 1로 변환할 때 최적의 임계값(threshold)을 찾아야 합니다.

In [ ]:
# 7-1. 멀티시드 앙상블로 확률 예측
print("=" * 50)
print("🎯 멀티시드 앙상블 및 최적 임계값 탐색")
print("=" * 50)

seeds = [42, 123, 456, 789, 1004]
best_xgb_params = xgb_search.best_params_
best_lgb_params = lgb_search.best_params_
best_cat_params = cat_search.best_params_
best_rf_params = rf_search.best_params_

# Validation 예측 수집
val_preds = {'xgb': [], 'lgb': [], 'cat': [], 'rf': []}

for seed in seeds:
    # XGBoost
    xgb_m = XGBClassifier(**best_xgb_params, random_state=seed, verbosity=0, use_label_encoder=False, eval_metric='auc')
    xgb_m.fit(X_train_scaled, y_train)
    val_preds['xgb'].append(xgb_m.predict_proba(X_val_scaled)[:, 1])
    
    # LightGBM
    lgb_m = LGBMClassifier(**best_lgb_params, random_state=seed, verbose=-1)
    lgb_m.fit(X_train_scaled, y_train)
    val_preds['lgb'].append(lgb_m.predict_proba(X_val_scaled)[:, 1])
    
    # CatBoost
    cat_m = CatBoostClassifier(**best_cat_params, random_state=seed, verbose=0)
    cat_m.fit(X_train_scaled, y_train)
    val_preds['cat'].append(cat_m.predict_proba(X_val_scaled)[:, 1])
    
    # Random Forest
    rf_m = RandomForestClassifier(**best_rf_params, random_state=seed, n_jobs=-1)
    rf_m.fit(X_train_scaled, y_train)
    val_preds['rf'].append(rf_m.predict_proba(X_val_scaled)[:, 1])

# 각 모델 평균
pred_xgb = np.mean(val_preds['xgb'], axis=0)
pred_lgb = np.mean(val_preds['lgb'], axis=0)
pred_cat = np.mean(val_preds['cat'], axis=0)
pred_rf = np.mean(val_preds['rf'], axis=0)

print("멀티시드 학습 완료!")

In [ ]:
# 7-2. 최적 가중치 탐색
best_score = 0
best_weights = None

for w1 in np.arange(0.1, 0.6, 0.05):
    for w2 in np.arange(0.1, 0.6, 0.05):
        for w3 in np.arange(0.1, 0.6, 0.05):
            w4 = round(1 - w1 - w2 - w3, 2)
            if w4 >= 0.05:
                prob = w1*pred_xgb + w2*pred_lgb + w3*pred_cat + w4*pred_rf
                score = roc_auc_score(y_val, prob)
                if score > best_score:
                    best_score = score
                    best_weights = (w1, w2, w3, w4)

print(f"\n🏆 최적 가중치: XGB={best_weights[0]:.2f}, LGB={best_weights[1]:.2f}, CAT={best_weights[2]:.2f}, RF={best_weights[3]:.2f}")
print(f"🎯 가중 앙상블 Val AUC: {best_score:.5f}")

In [ ]:
# 7-3. 최적 임계값(Threshold) 탐색 ⭐⭐⭐
print("\n" + "=" * 50)
print("🔍 최적 임계값(Threshold) 탐색")
print("=" * 50)

# 앙상블 확률
w1, w2, w3, w4 = best_weights
val_prob = w1*pred_xgb + w2*pred_lgb + w3*pred_cat + w4*pred_rf

# 다양한 임계값 테스트
best_threshold = 0.5
best_f1 = 0
best_acc = 0

print("\nThreshold | Accuracy | F1-Score | AUC")
print("-" * 45)

for thresh in np.arange(0.3, 0.7, 0.02):
    pred_label = (val_prob >= thresh).astype(int)
    acc = accuracy_score(y_val, pred_label)
    f1 = f1_score(y_val, pred_label)
    
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = thresh
        best_acc = acc
    
    if thresh in [0.40, 0.45, 0.50, 0.55, 0.60]:
        print(f"  {thresh:.2f}    |  {acc:.4f}  |  {f1:.4f}   | {best_score:.4f}")

print("-" * 45)
print(f"\n🏆 최적 임계값: {best_threshold:.2f}")
print(f"   Accuracy: {best_acc:.4f}")
print(f"   F1-Score: {best_f1:.4f}")

## 📌 STEP 8: 최종 예측 (0 또는 1 정수)

In [ ]:
# 8-1. 전체 데이터로 최종 학습
print("=" * 50)
print("📝 최종 모델 학습")
print("=" * 50)

final_preds = {'xgb': [], 'lgb': [], 'cat': [], 'rf': []}

for seed in seeds:
    print(f"Seed {seed} 학습 중...")
    
    xgb_m = XGBClassifier(**best_xgb_params, random_state=seed, verbosity=0, use_label_encoder=False, eval_metric='auc')
    xgb_m.fit(X_full_scaled, y)
    final_preds['xgb'].append(xgb_m.predict_proba(X_test_final)[:, 1])
    
    lgb_m = LGBMClassifier(**best_lgb_params, random_state=seed, verbose=-1)
    lgb_m.fit(X_full_scaled, y)
    final_preds['lgb'].append(lgb_m.predict_proba(X_test_final)[:, 1])
    
    cat_m = CatBoostClassifier(**best_cat_params, random_state=seed, verbose=0)
    cat_m.fit(X_full_scaled, y)
    final_preds['cat'].append(cat_m.predict_proba(X_test_final)[:, 1])
    
    rf_m = RandomForestClassifier(**best_rf_params, random_state=seed, n_jobs=-1)
    rf_m.fit(X_full_scaled, y)
    final_preds['rf'].append(rf_m.predict_proba(X_test_final)[:, 1])

print("\n✅ 최종 학습 완료!")

In [ ]:
# 8-2. 앙상블 확률 계산
test_xgb = np.mean(final_preds['xgb'], axis=0)
test_lgb = np.mean(final_preds['lgb'], axis=0)
test_cat = np.mean(final_preds['cat'], axis=0)
test_rf = np.mean(final_preds['rf'], axis=0)

# 가중 평균
test_prob = w1*test_xgb + w2*test_lgb + w3*test_cat + w4*test_rf

print(f"확률값 범위: {test_prob.min():.4f} ~ {test_prob.max():.4f}")
print(f"확률값 평균: {test_prob.mean():.4f}")

In [ ]:
# 8-3. 최적 임계값으로 0/1 변환 ⭐⭐⭐
print(f"\n🎯 임계값 {best_threshold:.2f} 적용하여 0/1 변환")

# 확률 → 0 또는 1 정수로 변환
final_prediction = (test_prob >= best_threshold).astype(int)

print(f"\n예측 결과 분포:")
print(f"  0 (비흡연): {(final_prediction == 0).sum()}명")
print(f"  1 (흡연):   {(final_prediction == 1).sum()}명")
print(f"  흡연 비율:  {final_prediction.mean()*100:.2f}%")

## 📌 STEP 9: 제출 파일 생성

In [ ]:
# 9-1. 제출 파일 생성
submission_df = submission.copy()
submission_df['label'] = final_prediction

# label이 정수인지 확인
submission_df['label'] = submission_df['label'].astype(int)

print("📋 제출 파일 미리보기:")
display(submission_df.head(10))

print(f"\nlabel 데이터 타입: {submission_df['label'].dtype}")
print(f"label 고유값: {submission_df['label'].unique()}")

In [ ]:
# 9-2. 파일 저장
output_path = result_path + 'submission_v2_final.csv'
submission_df.to_csv(output_path, index=False)

print(f"\n✅ 저장 완료: {output_path}")

In [ ]:
# 9-3. 제출 파일 검증
print("\n🔍 제출 파일 검증:")
print(f"   행 개수: {len(submission_df)}")
print(f"   컬럼: {submission_df.columns.tolist()}")
print(f"   label 타입: {submission_df['label'].dtype}")
print(f"   label 값: {sorted(submission_df['label'].unique())}")
print(f"   결측치: {submission_df['label'].isnull().sum()}")

# 검증
if submission_df['label'].dtype in ['int64', 'int32'] and set(submission_df['label'].unique()).issubset({0, 1}):
    print("\n✅ 검증 통과! 0과 1 정수값으로 되어 있습니다.")
else:
    print("\n⚠️ 검증 실패! label 값을 확인하세요.")

## 📌 STEP 10: 파일 다운로드

In [ ]:
# 10-1. Colab에서 다운로드
from google.colab import files

files.download(output_path)

print("\n" + "=" * 50)
print("🎉 완료!")
print("=" * 50)
print(f"\n📥 다운로드된 파일: submission_v2_final.csv")
print(f"\n📊 예측 결과:")
print(f"   - 비흡연(0): {(final_prediction == 0).sum()}명")
print(f"   - 흡연(1):   {(final_prediction == 1).sum()}명")
print(f"\n🎯 Validation 성능:")
print(f"   - AUC: {best_score:.5f}")
print(f"   - Accuracy: {best_acc:.4f}")
print(f"   - F1-Score: {best_f1:.4f}")
print(f"   - Threshold: {best_threshold:.2f}")
print("\n이 파일을 해커톤 사이트에 제출하세요!")